# Session 1 — Survival Python

**DS4Eng · Applied Data Science for Engineering · Organisation Lab**
Wednesday 16 September 2026 · Alfons Marquès

---

## From a spreadsheet to a DataFrame

A **DataFrame** — a table held in Python's memory, rows and columns like a sheet — is
where every session of this lab starts. By the end of these two hours you will have
**loaded a real industrial dataset into one, looked at it properly, and asked it one
honest question**.

You do not need to know Python to start. You need to read carefully and to be
suspicious of what you are told.

## How this notebook works

A **notebook** — this page: a stack of text cells and code cells that you run in
order — is the only tool of the session. This one has **three zones**. You only ever
work in the third one.

| Zone | What it is | Do you touch it? |
|---|---|---|
| ⚙️ **CONFIGURE** | The few settings of the session | **Yes** — this is the only place |
| 🔧 **ENGINE** | Helper functions that do the plumbing | **No** — run it and forget it |
| 🔬 **WORK** | Where you actually do things | **Yes** — this is the session |

And the work comes in **three tiers**:

1. **Guided** — we all do it together, step by step. Nobody is left behind.
2. **Semi-guided** — a task and a hint. In pairs.
3. **Open** — no hints. You decide, and you explain to your pair how you checked.

> **If you get stuck, do not sit there.** Every task has a folded answer right
> underneath it. Open it, move on, and ask afterwards. Losing 40 minutes on one
> cell helps nobody.

## How to use this notebook in Colab

If you have never used a notebook before, this is all you need.

1. **Save your own copy first.** `File → Save a copy in Drive`. Until you do this you
   are looking at the shared original, and your edits will vanish.
2. **A notebook is a stack of cells.** Grey cells are code; the others are text.
   Click a code cell and press **Shift+Enter**: it runs, its output appears
   underneath, and the cursor moves to the next cell.
3. **Run from the top, in order.** Later cells depend on earlier ones. If you see
   `NameError: name 'df' is not defined`, you skipped one — use `Runtime → Run all`.
4. **Colab forgets.** After about 90 idle minutes it recycles the machine and every
   variable disappears. Nothing is lost: `Runtime → Run all` rebuilds everything from
   the data in one click. The notebook is the source of truth, not the memory.

Nothing to install and nothing to upload: the data is fetched from the course
repository by the first code cell.

## Words we will use today

Plain meanings first, so nobody has to guess.

- **DataFrame** — a table held in Python's memory. Rows and columns, like a sheet.
- **row** — one observation. Here: one production cycle of a machine.
- **column** — one thing that was measured.
- **target** — the column you are trying to explain or predict.
- **rate** — a count divided by a total. Between 0 and 1.
- **flag** — a column that is either 0 or 1: a yes/no answer stored as a number.
- **notebook** — this page: a stack of text cells and code cells that you run in order.

---

## ⚙️ CONFIGURE

Everything the session depends on, in one place. Run it.

In [1]:
# Where the data lives. It is a public file — nothing to install, nothing to upload.
DATA_URL = "https://raw.githubusercontent.com/DATANINJA-dev/ds4eng-lab/main/data/ai4i2020.csv"

# The column we care about today.
TARGET = "Machine failure"

# The five columns that record *why* a machine failed.
FAILURE_FLAGS = ["TWF", "HDF", "PWF", "OSF", "RNF"]

## 🔧 ENGINE

Plumbing. Run it once and forget it. **Do not change anything in here.**

In [2]:
import io
import urllib.request

import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 50)


def load_data(url=None):
    """Download the dataset and hand it back as a DataFrame."""
    url = url or DATA_URL
    raw = urllib.request.urlopen(url, timeout=60).read()
    buffer = io.BytesIO(raw)
    if url.endswith(".parquet"):
        return pd.read_parquet(buffer)
    return pd.read_csv(buffer)


def pct(part, whole):
    """Turn a count into a percentage, rounded to two decimals."""
    return round(100 * part / whole, 2)


print("Engine ready.")

Engine ready.


---

# 🔬 Tier 1 — Guided · ~15 min

We do this together. Run each cell, then read what came out **before** running the
next one. The whole skill today is looking at output on purpose.

### 1.1 · Load the table

One line. No installing, no uploading, no dragging files into the sidebar.

In [3]:
df = load_data()
df.shape

(10000, 14)

`shape` gives you `(rows, columns)`.

**Read the number of columns and remember it.** We are coming back to it in about
ninety seconds.

### 1.2 · Look at the thing

Never analyse a table you have not looked at.

In [4]:
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


### 1.3 · What kind of data is in each column?

In [5]:
df.dtypes

UDI                          int64
Product ID                  object
Type                        object
Air temperature [K]        float64
Process temperature [K]    float64
Rotational speed [rpm]       int64
Torque [Nm]                float64
Tool wear [min]              int64
Machine failure              int64
TWF                          int64
HDF                          int64
PWF                          int64
OSF                          int64
RNF                          int64
dtype: object

Three things worth noticing, and none of them are about Python:

- The temperatures are in **kelvin**, not celsius. 298 K is about 25 °C.
- **Tool wear** is in minutes — it is a running total, not an instant measurement.
- **Type** is a letter (L, M, H): the quality variant of the product. It is a
  category wearing the costume of a text column.

### 1.4 · The summary you should never trust on its own

In [6]:
df.describe()

,UDI,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
count,10000.00000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000
mean,5000.50000,300.004930,310.005560,1538.776100,39.986910,107.951000,0.033900,0.004600,0.011500,0.009500,0.009800,0.00190
std,2886.89568,2.000259,1.483734,179.284096,9.968934,63.654147,0.180981,0.067671,0.106625,0.097009,0.098514,0.04355
min,1.00000,295.300000,305.700000,1168.000000,3.800000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
25%,2500.75000,298.300000,308.800000,1423.000000,33.200000,53.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
50%,5000.50000,300.100000,310.100000,1503.000000,40.100000,108.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
75%,7500.25000,301.500000,311.100000,1612.000000,46.800000,162.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
max,10000.00000,304.500000,313.800000,2886.000000,76.600000,253.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000


`describe()` only speaks about numeric columns, and it happily computes the mean of
things that have no business having a mean — like the 0/1 flags. It is a starting
point, not an answer.

---

### ✋ Checkpoint

The official dataset card for AI4I 2020 describes **6 variables**.
How many columns did `shape` report?

<details>
<summary><b>Open this once you have an answer</b></summary>

**14, not 6.**

The card counts the *interesting* ones and quietly omits `UDI`, `Product ID`, `Type`
and the five failure flags. Neither number is a lie — they are answering different
questions.

This is the first lesson of the course, and it is not a Python lesson:
**the documentation describes what someone meant to publish; the data is what
actually arrived.** You check the second one against the first, every time.

</details>

---

# 🔬 Tier 2 — Semi-guided · ~20 min · **in pairs**

Pair up — ideally with someone whose background is not yours. One drives, one reads
the output aloud. Swap after each task.

Each task has a hint. Each task has a folded answer. Use them in that order.

### Task 2.1 · How often does a machine actually fail?

> **Hint** — `TARGET` is stored as 0 and 1. The average of a column of zeros and
> ones *is* the rate of ones. `df[TARGET].mean()` and the `pct()` helper are enough.

In [7]:
# your code here

<details>
<summary><b>Answer 2.1</b></summary>

```python
failures = df[TARGET].sum()
print("failures:", failures, "of", len(df))
print("failure rate:", pct(failures, len(df)), "%")
```

**339 failures out of 10 000 — about 3.4 %.**

Park that number somewhere. In week 9, when a model announces that it is 96.6 %
accurate, you will want to remember that predicting "nothing ever breaks" scores
exactly that.

</details>

### Task 2.2 · Does the product variant change anything?

There are three quality variants in `Type`: L, M and H. Do they fail at the same rate?

> **Hint** — `df.groupby("Type")[TARGET]` and then `.mean()` gives you a rate per
> group. Add `.count()` if you want to know how many of each variant there are.

In [8]:
# your code here

<details>
<summary><b>Answer 2.2</b></summary>

```python
by_type = df.groupby("Type")[TARGET].agg(["count", "sum", "mean"])
by_type["rate_%"] = (100 * by_type["mean"]).round(2)
by_type
```

| Type | machines | failures | rate |
|---|---|---|---|
| **L** (low) | 6 000 | 235 | **3.92 %** |
| **M** (medium) | 2 997 | 83 | **2.77 %** |
| **H** (high) | 1 003 | 21 | **2.09 %** |

The cheap variant fails **almost twice as often** as the premium one. That is an
engineering finding, not a programming one — and it is the kind of sentence that
actually reaches a production meeting.

Now the caveat, and say it out loud when you report this: the variants are **very**
unevenly represented. The H number rests on a thousand machines, not ten thousand.
That does not make it wrong. It makes it weaker evidence, and hiding that is how
analyses lose their credibility.

</details>

### Task 2.3 · Why do they fail?

The five columns in `FAILURE_FLAGS` record the *cause*: tool wear (TWF), heat
dissipation (HDF), power (PWF), overstrain (OSF) and random (RNF).

How many times does each one fire?

> **Hint** — you can select several columns at once with `df[FAILURE_FLAGS]`, and
> `.sum()` on that gives you one total per column.

In [9]:
# your code here

<details>
<summary><b>Answer 2.3</b></summary>

```python
df[FAILURE_FLAGS].sum()
```

TWF 46 · HDF 115 · PWF 95 · OSF 98 · RNF 19 — **373 in total.**

Now add that up and compare it with the 339 failures from task 2.1.

They do not match. Hold that thought — it is the whole of Tier 3.

</details>

---

# 🔬 Tier 3 — Open · ~15 min · no hints

## The documentation makes a claim. Test it.

The UCI page for this dataset states, in writing:

> *"The machine failure label indicates whether the machine has failed in this
> particular datapoint for any of the following failure modes... if at least one of
> the above failure modes is true, the process fails and the `Machine failure` label
> is set to 1."*

That is a claim about the data. It is checkable in about four lines.

**Your job:** find out whether it is true, and if it is not, describe exactly how it
breaks. Show your answer to your pair and explain how you checked it.

Two questions to aim at:
1. Are there rows where a **flag is on but `Machine failure` is 0**?
2. Are there rows where **`Machine failure` is 1 but no flag is on**?

In [10]:
# your code here

<details>
<summary><b>Only open this after you have tried</b></summary>

```python
any_flag = df[FAILURE_FLAGS].sum(axis=1) > 0
failed = df[TARGET] == 1

print("flag on, but target says no failure:", (any_flag & ~failed).sum())
print("target says failure, but no flag on:", (failed & ~any_flag).sum())
print()
print(df[any_flag & ~failed][FAILURE_FLAGS].sum())
```

**The claim is false in 27 rows.**

- **18 rows** have a failure flag on while `Machine failure` is 0 — and all 18 are
  `RNF`, the random failures. Whoever built the target decided random failures do not
  count as failures. That decision is not written anywhere on the dataset page.
- **9 rows** are marked as failures with no cause recorded at all.

Neither group is an error you can "fix". They are a **decision someone made and did
not document**, and an **absence you cannot explain**. In week 4 you will meet 41 951
missing values in one file and this will feel familiar.

**The sentence to take home:** a target variable is not a fact. It is somebody's
definition, and it is your job to find out whose and what they decided.

</details>

---

## Ask it, then audit it · ~10 min

You will use a language model in this course. That is expected, it is declared, and
the subject's AI instructions say so explicitly. What is **not** optional is the
second half of the sentence.

**Do this now**, with Gemini:

1. Ask it to answer Tier 3 for you. Paste the column names and the question.
2. Run whatever it gives you.
3. Then answer these three questions — out loud, to your pair:

   - **What did it get right?**
   - **What did it assume that it had no way of knowing?**
   - **How did you check?**

We will repeat these three questions every single week. By the time you fill in the
authorship declaration on Atenea for your project, the box that asks *how you
validated the AI's output* will not be paperwork — it will be a description of what
you have been doing since September.

> The model writes the code. It does not make the judgement.
> **This course is about the judgement.**

---

## Before you go

**What we did:** loaded a real industrial dataset without installing anything, looked
at it properly, and caught its official documentation being wrong.

**What to bring next week:**
- Your own laptop
- A Google account you are actually logged into
- This notebook, saved to your own Drive (`File → Save a copy in Drive`)

**Next session — 23 September:** *Understanding the problem.* What you ask a dataset,
and what each column really means. We move to a garment factory: teams, standard
minute value, work-in-progress and productivity targets.

---

### Data

AI4I 2020 Predictive Maintenance Dataset. UCI Machine Learning Repository.
DOI: [10.24432/C5HS5C](https://doi.org/10.24432/C5HS5C) · Licence **CC BY 4.0**.